# RAG: Retrieval-Augmented Generation, built step by step

**Idea:** don't send everything. Prepare the documents once (**indexing**), then for each
question find only the few most relevant pieces (**retrieval**) and send those.

```
INDEXING (once)     raw files ─► 1 clean ─► 2 chunk ─► 3 embed ─► 4 store in vector DB
QUERY (each time)   question ─► embed ─► 5 retrieve top K ─► 6 prompt + Claude ─► answer
```

Every step saves its result in `notebooks/rag_output/`, so you can open it.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent            # the demos/ folder
DATA = ROOT / "data"                # the knowledge: the same files for all three demos
assert DATA.exists(), "Open this notebook from demos/notebooks/"
sys.path.append(str(ROOT))
import importlib, llm
importlib.reload(llm)               # picks up edits to llm.py when this cell is re-run
from llm import ask                 # small helper that calls Claude and shows the chat

import json, re, shutil
from html import unescape
import numpy as np
import pandas as pd

OUT = Path("rag_output")
for old in ["clean", "chunks.json", "embeddings.json"]:   # start from scratch
    shutil.rmtree(OUT / old, ignore_errors=True) if (OUT / old).is_dir() else (OUT / old).unlink(missing_ok=True)
OUT.mkdir(exist_ok=True)

def show(df):  # display a table with full, wrapped text
    return df.style.hide(axis="index").format(precision=3).set_properties(**{"white-space": "pre-wrap", "text-align": "left"})

## 0. The raw data

Real documents are messy. Here are 5 files in 3 formats: an intranet **HTML** page,
a **PDF** exported to text, and **Markdown** wiki pages.

In [ ]:
pd.DataFrame({"file": [p.name for p in sorted(DATA.iterdir())],
              "characters": [p.stat().st_size for p in sorted(DATA.iterdir())]})

In [ ]:
print((DATA / "leave-policy.html").read_text(encoding="utf-8")[:1200])

In [ ]:
print((DATA / "it-handbook.txt").read_text(encoding="utf-8"))

## 1. Clean

Remove everything that is not knowledge, so it does not end up in the chunks:
menus, cookie banners, footers, scripts, page headers, words broken across lines,
extra spaces and duplicate lines. Keep the headings: chunking will use them.

In [ ]:
from collections import Counter

BOILERPLATE = [r"^CONFIDENTIAL\b", r"^IT HANDBOOK\s+Page \d+", r"^> Last edited by"]

def html_to_text(raw, removed):
    raw, n = re.subn(r"<(head|script|style|nav|footer)\b.*?</\1>", "", raw, flags=re.S)  # page furniture
    raw, m = re.subn(r'<(div|p) class="(cookie-banner|meta)".*?</\1>', "", raw, flags=re.S)
    removed["page blocks (head, menu, banner, footer)"] += n + m
    removed["HTML tags"] += len(re.findall(r"<[^>]+>", raw))
    raw = re.sub(r"<h1[^>]*>", "\n# ", raw)            # keep headings as Markdown headings
    raw = re.sub(r"<h2[^>]*>", "\n## ", raw)
    raw = re.sub(r"<(p|li|br)[^>]*>", "\n", raw)       # paragraphs become lines
    raw = re.sub(r"<[^>]+>", "", raw)                  # drop every other tag
    return unescape(raw).replace("\xa0", " ")          # &nbsp; &#39; -> real characters

def pdf_text_to_text(raw, removed):
    raw, n = re.subn(r"-\n(\w)", r"\1", raw)           # "re-\nturned" -> "returned"
    removed["broken words fixed"] += n
    out, prev_blank, first = [], True, True
    for line in map(str.strip, raw.splitlines()):
        if not line:
            out.append("")
        elif prev_blank and len(line) < 40 and not line.endswith("."):
            out.append(("# " if first else "## ") + line)   # short line after a gap = a title
            first = False
        elif out and out[-1] and not out[-1].startswith("#") and not out[-1].endswith("."):
            out[-1] += " " + line                           # unfinished sentence: join the lines
            removed["wrapped lines joined"] += 1
        else:
            out.append(line)
        prev_blank = not line
    return "\n".join(out)

def clean(path):
    removed = Counter()
    lines = path.read_text(encoding="utf-8").splitlines()
    kept = [l for l in lines if not any(re.match(p, l.strip()) for p in BOILERPLATE)]
    removed["page headers / edit stamps"] = len(lines) - len(kept)
    text = "\n".join(kept)
    if path.suffix == ".html":
        text = html_to_text(text, removed)
    elif path.suffix == ".txt":
        text = pdf_text_to_text(text, removed)
    result, seen = [], set()
    for line in text.splitlines():
        line = re.sub(r"\s+", " ", line).strip()       # collapse spaces
        if not line:
            continue
        if line in seen:                               # drop duplicate lines
            removed["duplicate lines"] += 1
            continue
        seen.add(line)
        result.append(("\n" if line.startswith("#") and result else "") + line)
    return "\n".join(result) + "\n", removed

In [ ]:
(OUT / "clean").mkdir(exist_ok=True)
clean_docs, report = {}, []
for path in sorted(DATA.iterdir()):
    if path.suffix in (".html", ".txt", ".md"):            # graph.csv is for the KAG demo
        text, removed = clean(path)
        clean_docs[path.stem + ".md"] = text
        (OUT / "clean" / (path.stem + ".md")).write_text(text, encoding="utf-8")
        raw_chars = path.stat().st_size
        report.append({"file": path.name, "raw chars": raw_chars, "clean chars": len(text),
                       "removed": f"{1 - len(text) / raw_chars:.0%}",
                       "what was cleaned": ", ".join(f"{n} {what}" for what, n in removed.items() if n)
                                           or "nothing, already clean"})
show(pd.DataFrame(report))

Compare with the raw files above:

In [ ]:
print(clean_docs["leave-policy.md"])
print(clean_docs["it-handbook.md"])

## 2. Chunk

Split each document into small pieces that can be searched on their own.

- Split by **section heading** first, so a chunk never mixes two topics.
- Then pack lines into chunks of at most `MAX_CHARS` characters.
- Repeat the last `OVERLAP` lines of the previous chunk, so a fact on a boundary is not cut off.
- Keep **metadata** (source file and section) with every chunk.

In [ ]:
MAX_CHARS = 300
OVERLAP = 1

chunks = []
for source, text in clean_docs.items():
    title, sections, section, lines = "", [], "", []
    for line in text.splitlines() + ["## "]:
        if line.startswith("#"):
            if lines:
                sections.append((section, lines))
            if line.startswith("# "):
                title = line[2:]
            section, lines = line.lstrip("# "), []
        elif line.strip():
            lines.append(line)
    n = 0
    for section, lines in sections:
        current = []
        for line in lines:
            if current and len(" ".join(current + [line])) > MAX_CHARS:
                chunks.append({"id": f"{source[:-3]}-{n}", "source": source,
                               "section": f"{title} > {section}", "text": "\n".join(current)})
                n += 1
                current = current[-OVERLAP:]
            current.append(line)
        chunks.append({"id": f"{source[:-3]}-{n}", "source": source,
                       "section": f"{title} > {section}", "text": "\n".join(current)})
        n += 1

(OUT / "chunks.json").write_text(json.dumps(chunks, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"{len(chunks)} chunks")
show(pd.DataFrame(chunks).assign(chars=lambda d: d.text.str.len())[["id", "section", "chars", "text"]])

## 3. Embed

An **embedding model** turns each chunk into a vector: a list of numbers that captures
its *meaning*. Texts with similar meaning get vectors that point in a similar direction.

We use `BAAI/bge-small-en-v1.5`, a small open model that runs on this laptop (no API).

In [ ]:
import os, tempfile, warnings
os.environ.setdefault("HF_HUB_DISABLE_SYMLINKS_WARNING", "1")
warnings.filterwarnings("ignore", message="IProgress not found")
from fastembed import TextEmbedding
from loguru import logger
logger.disable("fastembed")

model = TextEmbedding("BAAI/bge-small-en-v1.5",
                      cache_dir=str(Path(os.environ.get("LOCALAPPDATA", tempfile.gettempdir())) / "fastembed_models"))
vectors = np.array(list(model.embed([c["text"] for c in chunks])))
print("shape:", vectors.shape, " -> one row of", vectors.shape[1], "numbers per chunk")
print(chunks[0]["text"][:60], "...")
print(vectors[0][:10].round(3), "...")

_ = (OUT / "embeddings.json").write_text(json.dumps(
    [{"id": c["id"], "vector": v.round(6).tolist()} for c, v in zip(chunks, vectors)]), encoding="utf-8")

**Why embeddings?** Similar *meaning* scores high even with *different words*:

In [ ]:
def cosine(a, b):
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))

q, same, other = model.embed(["Can I roll over unused vacation days?",
                              "Up to 5 unused leave days can be carried over into the next year.",
                              "Passwords must be at least 14 characters long."])
print(f"roll over vacation  vs  leave carried over : {cosine(q, same):.2f}")
print(f"roll over vacation  vs  passwords          : {cosine(q, other):.2f}")

A map of all chunks: the 384 numbers squeezed down to 2 (PCA). Chunks about the same topic sit together.

In [ ]:
import matplotlib.pyplot as plt

mean = vectors.mean(axis=0)
_, _, axes = np.linalg.svd(vectors - mean, full_matrices=False)
project = lambda v: (np.atleast_2d(v) - mean) @ axes[:2].T
points = project(vectors)
sources = sorted({c["source"] for c in chunks})

def draw_map(question=None, hit_ids=()):
    fig, ax = plt.subplots(figsize=(11, 7))
    for i, src in enumerate(sources):
        idx = [k for k, c in enumerate(chunks) if c["source"] == src]
        ax.scatter(points[idx, 0], points[idx, 1], s=80, color=f"C{i}", label=src)
    for k, c in enumerate(chunks):
        hit = c["id"] in hit_ids
        ax.annotate(c["section"].split(" > ")[1], points[k], fontsize=8, xytext=(5, 3),
                    textcoords="offset points", weight="bold" if hit else "normal")
        if hit:
            ax.scatter(*points[k], s=300, facecolors="none", edgecolors="black", linewidths=2)
    if question:
        qp = project(next(model.query_embed([question])))[0]
        ax.scatter(*qp, marker="*", s=400, color="black", label="question")
        for k, c in enumerate(chunks):
            if c["id"] in hit_ids:
                ax.plot([qp[0], points[k, 0]], [qp[1], points[k, 1]], "k--", lw=1)
    ax.legend(fontsize=8, loc="best"); ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(f"Question: {question}" if question else "Every chunk, placed by meaning")
    plt.show()

def draw_target(question, hit_ids=()):
    """The question in the centre; each chunk's distance from it is EXACT (1 - similarity, all 384 numbers).
    Only the angle is free: chunks are grouped by document. Shaded = the chunks retrieval returned."""
    from matplotlib.lines import Line2D
    q = next(model.query_embed([question]))
    sims = vectors @ q / (np.linalg.norm(vectors, axis=1) * np.linalg.norm(q))
    dist = 1 - sims
    hit = [c["id"] in hit_ids for c in chunks]
    fig = plt.figure(figsize=(10, 10))
    ax = fig.add_subplot(projection="polar")
    if any(hit):
        edge = max(d for d, h in zip(dist, hit) if h) + 0.003   # just past the last retrieved chunk
        circle = np.linspace(0, 2 * np.pi, 200)
        ax.fill(circle, np.full_like(circle, edge), color="gold", alpha=0.2, linewidth=0, zorder=0)
    slice_ = 2 * np.pi / len(sources)
    for s, src in enumerate(sources):
        idx = [k for k, c in enumerate(chunks) if c["source"] == src]
        angles = s * slice_ + slice_ * (np.arange(len(idx)) + 0.5) / len(idx)
        for a, k in zip(angles, idx):
            ax.scatter(a, dist[k], s=260 if hit[k] else 90, color=f"C{s}", zorder=3,
                       edgecolors="black" if hit[k] else "none", linewidths=2)
            ax.annotate(f"{chunks[k]['section'].split(' > ')[1]}  {sims[k]:.2f}", (a, dist[k]), fontsize=8,
                        xytext=(6, 4), textcoords="offset points", weight="bold" if hit[k] else "normal")
    ax.scatter(0, 0, marker="*", s=600, color="black", zorder=4)
    rings = np.arange(0.2, dist.max() + 0.1, 0.1)
    ax.set_rticks(rings); ax.set_yticklabels([f"{1 - r:.1f}" for r in rings], fontsize=8, color="gray")
    ax.set_rlabel_position(90); ax.set_xticks([]); ax.set_ylim(0, dist.max() + 0.05); ax.grid(alpha=0.4)
    handles = [Line2D([], [], marker="o", ls="", color=f"C{s}", label=src) for s, src in enumerate(sources)]
    handles += [Line2D([], [], marker="*", ls="", color="black", markersize=14, label="question"),
                Line2D([], [], marker="o", ls="", color="gold", alpha=0.5, markersize=14, label=f"retrieved (top {len(hit_ids)})")]
    ax.legend(handles=handles, fontsize=8, loc="upper left", bbox_to_anchor=(1.02, 1.0))
    ax.set_title(f"Question: {question}\nCloser to the centre = more similar (ring labels = similarity)", fontsize=11, pad=18)
    plt.show()

draw_map()

## 4. Store in a vector database

A vector database stores each chunk's **vector**, **text** and **metadata** together, and
can quickly find the vectors nearest to a new one. We use **Chroma**, which runs locally
and saves to `rag_output/chroma/`.

In [ ]:
import chromadb

client = chromadb.PersistentClient(path=str(OUT / "chroma"))
if "company_docs" in [c.name for c in client.list_collections()]:
    client.delete_collection("company_docs")                  # rebuild from scratch
db = client.create_collection("company_docs", embedding_function=None,
                              configuration={"hnsw": {"space": "cosine"}})
db.add(ids=[c["id"] for c in chunks],
       embeddings=vectors.tolist(),
       documents=[c["text"] for c in chunks],
       metadatas=[{"source": c["source"], "section": c["section"]} for c in chunks])
print("rows in the database:", db.count())

One row, exactly as stored:

In [ ]:
row = db.get(ids=[chunks[0]["id"]], include=["documents", "metadatas", "embeddings"])
{"id": row["ids"][0], "metadata": row["metadatas"][0], "document": row["documents"][0],
 "embedding": f"{row['embeddings'][0][:5].round(3).tolist()} ... ({len(row['embeddings'][0])} numbers)"}

### Look inside the database, like a SQL client

Chroma saves everything in **one SQLite file**: `rag_output/chroma/chroma.sqlite3`.
We open it **read-only** and run plain SQL: first each table on its own, then the joins that
put a chunk back together. Every query ends with `LIMIT 5` to keep the output short.

For fast search, Chroma also builds an **HNSW index** from the vectors, in the folder next to that file
(named with a long id). That index is what `db.query()` uses. In a real project you never query these
tables yourself; this is only to show there is no magic inside.

**Table 1: `embeddings`** (19 rows, one per chunk). The chunk's identity card:
- `id`: Chroma's internal row number, used by the other tables to point here
- `embedding_id`: our chunk id, such as `expenses-policy-0`
- `seq_id`: links to the row in `embeddings_queue` that holds the vector
- `segment_id`: which part of the collection the row belongs to

In [ ]:
import sqlite3

con = sqlite3.connect(f"{(OUT / 'chroma' / 'chroma.sqlite3').resolve().as_uri()}?mode=ro", uri=True)

def sql(query):
    return show(pd.read_sql_query(query, con))

sql("SELECT * FROM embeddings LIMIT 5")

**Table 2: `embedding_metadata`** (57 rows: 3 per chunk). The chunk's metadata and text, stored as
**key/value rows**: `source`, `section`, and `chroma:document` (the chunk text itself).
`id` points to `embeddings.id`. Each value goes in the column for its type, so here only `string_value` is filled.

In [ ]:
sql("SELECT * FROM embedding_metadata LIMIT 5")

**Table 3: `embeddings_queue`** (19 rows: one per write). Chroma's write log:
- `operation` 0 means "add"
- `vector`: the embedding as raw bytes (384 numbers × 4 bytes = 1,536 bytes). Raw bytes can't be printed, so the query shows their size and the first bytes in hex
- `metadata`: a JSON copy of the same metadata

(The `topic` column, a long internal address of the collection, is left out.)

In [ ]:
sql("""
SELECT seq_id,
       created_at,
       operation,
       id,
       encoding,
       length(vector)                      AS vector_bytes,
       hex(substr(vector, 1, 8)) || '…'    AS vector_first_bytes,
       metadata
FROM   embeddings_queue
LIMIT  5
""")

### Joining the tables

**The chunks.** `embeddings` joined to `embedding_metadata` once per key, turning the key/value rows
back into one row per chunk.

In [ ]:
sql("""
SELECT e.embedding_id    AS id,
       src.string_value  AS source,
       sec.string_value  AS section,
       doc.string_value  AS document
FROM   embeddings e
JOIN   embedding_metadata src ON src.id = e.id AND src.key = 'source'
JOIN   embedding_metadata sec ON sec.id = e.id AND sec.key = 'section'
JOIN   embedding_metadata doc ON doc.id = e.id AND doc.key = 'chroma:document'
ORDER  BY e.id
LIMIT  5
""")

**The embeddings.** `embeddings` joined to `embeddings_queue` on `seq_id`, with the raw bytes
decoded back into numbers. The `vector` column shows all 384 numbers (rounded to 4 decimals).
`length` shows every vector has length 1.0: they differ only in direction, and direction is the meaning.

In [ ]:
vec = pd.read_sql_query("""
SELECT e.embedding_id      AS id,
       q.encoding,
       length(q.vector)    AS bytes,
       q.vector
FROM   embeddings e
JOIN   embeddings_queue q ON q.seq_id = e.seq_id
ORDER  BY e.id
LIMIT  5
""", con)
vec["vector"] = vec.vector.map(lambda b: np.frombuffer(b, dtype=np.float32))   # bytes -> 384 numbers
vec["numbers"] = vec.vector.map(len)
vec["length"] = vec.vector.map(lambda v: round(float(np.linalg.norm(v)), 3))     # always 1.0
vec["vector"] = vec.vector.map(lambda v: v.astype(float).round(4).tolist())       # all 384, 4 decimals
show(vec[["id", "encoding", "bytes", "numbers", "length", "vector"]])

What SQL **cannot** do here: find the rows *closest in meaning* to a question.
`WHERE` matches exact values; meaning needs a distance between vectors.
That is the vector index's job, and it's the next step.

## 5. Retrieve

Embed the question with the **same** model, then ask the database for the `TOP_K` nearest chunks.

In [ ]:
TOP_K = 3

def retrieve(question, k=TOP_K):
    q_vec = next(model.query_embed([question])).tolist()
    res = db.query(query_embeddings=[q_vec], n_results=k, include=["documents", "metadatas", "distances"])
    return [{"id": i, "similarity": round(1 - d, 2), "section": m["section"], "text": t}
            for i, t, m, d in zip(res["ids"][0], res["documents"][0], res["metadatas"][0], res["distances"][0])]

question = "Can I roll over unused vacation days?"
hits = retrieve(question)
show(pd.DataFrame(hits))

Two views of the same search:
- **The map** shows neighbourhoods of meaning, but it is squeezed from 384 numbers to 2, so its distances are only approximate.
- **The target** puts the question in the centre. Each chunk's distance from the centre is **exact** (the real similarity),
  only the angle is free. The shaded circle ends at the last chunk retrieval returned (black rings).
  A dot on its edge **without** a black ring tied with that chunk, but lost the cut: retrieval keeps exactly `TOP_K` chunks.

In [ ]:
draw_map(question, [h["id"] for h in hits])
draw_target(question, [h["id"] for h in hits])

## 6. Augment and generate

Paste only the retrieved chunks into the prompt and ask Claude.

In [ ]:
def rag_answer(question):
    hits = retrieve(question)
    context = "\n\n".join(f"[{h['section']}]\n{h['text']}" for h in hits)
    system = ("Answer the employee's question using only the context below. "
              "If the context is not enough, say what is missing.\n\nContext:\n" + context)
    return ask(system, question)

rag_answer(question);

## Try your own questions

In [ ]:
show(pd.DataFrame(retrieve("Where can I park my car?")))

In [ ]:
rag_answer("What is the hotel limit per night?");

## 7. Where RAG struggles: connected facts

*"Who approves Omar's leave carry-over?"* needs **three** facts from **two** documents:
1. Carry-over is approved by the line manager (leave policy)
2. Omar is on the Sales team (team directory, *People*)
3. Sara Ahmed is the line manager of the Sales team (team directory, *Teams*)

Fact 3 mentions neither Omar nor carry-over, so it does not look similar to the question.

In [ ]:
question = "Who approves Omar's leave carry-over?"
hits = retrieve(question)
print("Retrieved a chunk that mentions Sara?", any("Sara" in h["text"] for h in hits))
show(pd.DataFrame(hits))

Two views of the same search:
- **The map** shows neighbourhoods of meaning, but it is squeezed from 384 numbers to 2, so its distances are only approximate.
- **The target** puts the question in the centre. Each chunk's distance from the centre is **exact** (the real similarity),
  only the angle is free. The shaded circle ends at the last chunk retrieval returned (black rings).
  A dot on its edge **without** a black ring tied with that chunk, but lost the cut: retrieval keeps exactly `TOP_K` chunks.

In [ ]:
draw_map(question, [h["id"] for h in hits])
draw_target(question, [h["id"] for h in hits])

In [ ]:
rag_answer(question);

## Knobs to try live

- Change `MAX_CHARS` or `OVERLAP` in step 2, then rerun steps 2 to 5. Bigger chunks may catch
  the Sara line by luck, but each chunk gets less focused.
- Change `TOP_K` in step 5: more chunks, more chance, more noise and cost.

## Takeaways

- ✅ Scales to huge collections: each question only sends a few chunks.
- ✅ Embeddings find meaning, not just matching words.
- ❌ Quality depends on every step: cleaning, chunking, the embedding model, `TOP_K`.
- ❌ Weak on questions that need several connected facts (next: KAG).